# APIM ❤️ Microsoft Foundry

## End-to-End Private Lab
![architecture](images/foundry-e2e-private.gif)

This lab deploys an **end-to-end private** Microsoft Foundry environment fronted by `Azure API Management` (StandardV2) acting as the AI Gateway. All backend services (Foundry, AI Search, Cosmos DB, Storage, the cross-region OpenAI account, and APIM itself) sit behind private endpoints in a single VNet. A Windows jump-box VM accessible via `Azure Bastion` is included so you can test the private endpoints from inside the VNet.

### What gets deployed
- Microsoft Foundry account + project with `networkInjections` to a private agent subnet
- Standard backend resources (Azure AI Search, Cosmos DB, Storage) behind private endpoints with private DNS zones
- APIM `StandardV2` with outbound VNet integration **and** an inbound private endpoint
- An Azure OpenAI inference API imported into APIM, with a managed-identity inbound policy (see [policy.xml](policy.xml)) and a dedicated APIM subscription scoped to that API for direct-from-VNet testing
- An APIM gateway connection on the Foundry project so models can be called as `apim-gateway/<model-name>`
- A cross-region Azure OpenAI account with private endpoint into the primary VNet, exposed through APIM as `apim-gateway-crossregion/<model-name>` (also with a scoped APIM subscription for direct testing)
- Application Insights + Log Analytics, connected to the Foundry project for agent tracing
- Azure Bastion + Windows jump-box VM with desktop test scripts that call APIM directly using the scoped subscription keys (no caller-side Azure AD identity required)

### Notes
- Provisioning APIM `StandardV2` with VNet integration takes **~30 minutes**.
- The Foundry project endpoint is private. Run the test cells from the jump-box VM (via Bastion) once the deployment is complete.
- See the [clean-up notebook](clean-up-resources.ipynb) when you are done.

### Prerequisites
- [Python 3.12 or later](https://www.python.org/) installed
- [VS Code](https://code.visualstudio.com/) with the [Jupyter extension](https://marketplace.visualstudio.com/items?itemName=ms-toolsai.jupyter)
- [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli) installed and signed in
- An Azure subscription with `Contributor` permissions and quota for the chosen models in **both** the primary and cross-region locations

<a id='0'></a>
### 0️⃣ Initialize notebook variables

In [22]:
import os, sys, json
sys.path.insert(1, '../../shared')  # add the shared directory to the Python path
import utils

deployment_name = os.path.basename(os.path.dirname(globals()['__vsc_ipynb_file__']))
resource_group_name = f"lab-{deployment_name}-v7"
# West US 3 chosen because eastus2 is currently out of Azure AI Search capacity
# ('InsufficientResourcesAvailable'). Foundry private-agent (network-injected)
# capability hosts are supported in westus3, eastus2, eastus, australiaeast,
# uksouth, etc. Sweden Central can fail with the
# 'tags.internal.containerapps.resource-owner=AIAgents not allowed' error.
resource_group_location = 'westus3'
cross_region_location = 'eastus2'

# Primary model exposed through apim-gateway/<model>
model_name = 'gpt-4o-mini'
model_version = '2024-07-18'
model_capacity = 30
model_sku = 'GlobalStandard'

# Cross-region model exposed through apim-gateway-crossregion/<model>
deploy_cross_region = True
cross_region_model_name = 'gpt-4o'
cross_region_model_version = '2024-11-20'

# APIM
apim_sku = 'StandardV2'
publisher_email = 'apim-admin@contoso.com'
publisher_name = 'AI Foundry'
apim_connection_name = 'apim-gateway'
apim_cross_region_connection_name = 'apim-gateway-crossregion'
inference_api_version = '2024-10-21'

# Bastion + jumpbox
deploy_bastion = True
jumpbox_admin_username = 'azureuser'
jumpbox_admin_password = os.environ.get('JUMPBOX_PASSWORD', '@Aa123456789')   # set JUMPBOX_PASSWORD env var or change this default
# Pre-install Python 3.12, Azure CLI, Git, VS Code, PowerShell 7 and Windows Terminal
# on the jump-box, then clone Azure-Samples/AI-Gateway and pip install requirements.txt.
# Bootstrap runs on first boot and logs to C:\bootstrap.log on the VM (~5-10 min).
install_dev_tools = True

deploy_application_insights = True

# Auto-approve the AI Search → Foundry shared private link via a Bicep deployment script.
# Default is False because Microsoft.Resources/deploymentScripts requires shared-key storage
# access, which is blocked by tenant policy in some environments (e.g. KeyBasedAuthenticationNotPermitted).
# Set to True if your tenant allows it; otherwise the next cell approves the SPL manually.
auto_approve_spl = False

utils.print_ok('Notebook initialized')

✅ Notebook initialized ⌚ 22:36:29.838455 


<a id='1'></a>
### 1️⃣ Verify the Azure CLI and the connected Azure subscription

In [23]:
output = utils.run('az account show', 'Retrieved az account', 'Failed to get the current az account')

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']
    utils.print_info(f'Current user: {current_user}')
    utils.print_info(f'Tenant ID:    {tenant_id}')
    utils.print_info(f'Subscription: {subscription_id}')

⚙️ Running: az account show 
✅ Retrieved az account ⌚ 22:36:34.321579 :1s]
👉🏽 Current user: admin@MngEnvMCAP734518.onmicrosoft.com
👉🏽 Tenant ID:    cdfe81b5-821e-4f07-9ea7-516efc8497e4
👉🏽 Subscription: 3d2c527a-481d-4e13-b3a1-637924b33343


<a id='2'></a>
### 2️⃣ Create deployment using 🦾 Bicep

This deploys the full private topology. **Provisioning APIM StandardV2 with VNet integration takes ~30 minutes**, so the deployment as a whole typically completes in 35-45 minutes.

In [28]:
utils.create_resource_group(resource_group_name, resource_group_location)

bicep_parameters = {
    '$schema': 'https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#',
    'contentVersion': '1.0.0.0',
    'parameters': {
        'location':                       { 'value': resource_group_location },
        'locationCrossRegion':            { 'value': cross_region_location },
        'modelName':                      { 'value': model_name },
        'modelVersion':                   { 'value': model_version },
        'modelCapacity':                  { 'value': model_capacity },
        'modelSkuName':                   { 'value': model_sku },
        'apimSku':                        { 'value': apim_sku },
        'publisherEmail':                 { 'value': publisher_email },
        'publisherName':                  { 'value': publisher_name },
        'apimConnectionName':             { 'value': apim_connection_name },
        'inferenceApiVersion':            { 'value': inference_api_version },
        'deployCrossRegionOpenAI':        { 'value': deploy_cross_region },
        'crossRegionModelName':           { 'value': cross_region_model_name },
        'crossRegionModelVersion':        { 'value': cross_region_model_version },
        'apimCrossRegionConnectionName':  { 'value': apim_cross_region_connection_name },
        'deployApplicationInsights':      { 'value': deploy_application_insights },
        'deployBastion':                  { 'value': deploy_bastion },
        'jumpboxAdminUsername':           { 'value': jumpbox_admin_username },
        'jumpboxAdminPassword':           { 'value': jumpbox_admin_password },
        'installDevTools':                { 'value': install_dev_tools },
        'autoApproveSharedPrivateLink':   { 'value': auto_approve_spl },
    }
}

with open('params.json', 'w') as f:
    f.write(json.dumps(bicep_parameters))

output = utils.run(
    f'az deployment group create --name {deployment_name} --resource-group {resource_group_name} --template-file main.bicep --parameters params.json',
    f"Deployment '{deployment_name}' succeeded",
    f"Deployment '{deployment_name}' failed")

⚙️ Running: az group show --name lab-foundry-e2e-private-v7 
👉🏽 Using existing resource group 'lab-foundry-e2e-private-v7'
⚙️ Running: az deployment group create --name foundry-e2e-private --resource-group lab-foundry-e2e-private-v7 --template-file main.bicep --parameters params.json 
✅ Deployment 'foundry-e2e-private' succeeded ⌚ 23:12:03.211987 :20s]


<a id='3'></a>
### 3️⃣ Capture deployment outputs

In [27]:
output = utils.run(
    f'az deployment group show --name {deployment_name} -g {resource_group_name}',
    f"Retrieved deployment: {deployment_name}",
    f"Failed to retrieve deployment: {deployment_name}")

ai_project_endpoint = ''
apim_gateway_url = ''
apim_gateway_connection = ''
apim_cross_region_connection = ''
app_insights_connection_string = ''
jumpbox_name = ''
bastion_name = ''

if output.success and output.json_data:
    ai_services_name              = utils.get_deployment_output(output, 'aiServicesName',              'AI Services Name')
    ai_project_name               = utils.get_deployment_output(output, 'aiProjectName',               'AI Project Name')
    ai_project_endpoint           = utils.get_deployment_output(output, 'aiProjectEndpoint',           'AI Project Endpoint')
    apim_service_name             = utils.get_deployment_output(output, 'apimServiceName',             'APIM Service Name')
    apim_gateway_url              = utils.get_deployment_output(output, 'apimGatewayUrl',              'APIM Gateway URL')
    apim_gateway_connection       = utils.get_deployment_output(output, 'apimGatewayConnectionName',   'APIM Gateway Connection Name')
    apim_cross_region_connection  = utils.get_deployment_output(output, 'apimCrossRegionConnectionName', 'APIM Cross-Region Connection Name')
    app_insights_connection_string = utils.get_deployment_output(output, 'appInsightsConnectionString', 'App Insights Connection String', True)
    jumpbox_name                  = utils.get_deployment_output(output, 'jumpboxName',                 'Jumpbox VM Name')
    bastion_name                  = utils.get_deployment_output(output, 'bastionName',                 'Bastion Name')
    ai_search_name                = utils.get_deployment_output(output, 'aiSearchName',                'AI Search Name')
    shared_private_link_name      = utils.get_deployment_output(output, 'sharedPrivateLinkName',       'SPL Name')

⚙️ Running: az deployment group show --name foundry-e2e-private -g lab-foundry-e2e-private-v7 
✅ Retrieved deployment: foundry-e2e-private ⌚ 23:00:08.563461 :3s]
👉🏽 AI Services Name: aiserviceswaft
👉🏽 AI Project Name: projectwaft
👉🏽 AI Project Endpoint: https://aiserviceswaft.services.ai.azure.com/api/projects/projectwaft
👉🏽 APIM Service Name: aiserviceswaftapim
👉🏽 APIM Gateway URL: https://aiserviceswaftapim.azure-api.net
👉🏽 APIM Gateway Connection Name: apim-gateway
👉🏽 APIM Cross-Region Connection Name: apim-gateway-crossregion
👉🏽 App Insights Connection String: ****1cde
👉🏽 Jumpbox VM Name: waft-jumpbox
👉🏽 Bastion Name: aiserviceswaft-bastion
👉🏽 AI Search Name: aiserviceswaftsearch
👉🏽 SPL Name: search-to-aiservices-openai


<a id='3a'></a>
### 3️⃣.1 Verify the AI Search → Foundry Shared Private Link is Approved

If `auto_approve_spl = True` (default), the deployment script in the Bicep template already approved the connection. This cell verifies the status and approves it as a fallback if the auto-approve was disabled or blocked by tenant policy.

In [ ]:
import time

pec_name = None
for _ in range(30):
    out = utils.run(
        f'az network private-endpoint-connection list --id /subscriptions/{subscription_id}/resourceGroups/{resource_group_name}/providers/Microsoft.CognitiveServices/accounts/{ai_services_name} --query "[?starts_with(name, \'{shared_private_link_name}.\') || contains(properties.privateEndpoint.id, \'{shared_private_link_name}\')].name | [0]" -o tsv',
        'Checked pending PE connections', 'Failed to list PE connections')
    if out.success and out.text and out.text.strip() and out.text.strip() != 'None':
        pec_name = out.text.strip()
        # The list query returns the name as '<accountName>/<connection-name>'
        # for Cognitive Services accounts. Strip the prefix so we don't end up
        # with a duplicated account segment in the resource ID.
        if '/' in pec_name:
            pec_name = pec_name.split('/')[-1]
        break
    utils.print_info('PE connection not yet visible, sleeping 20s...')
    time.sleep(20)

if pec_name:
    pec_id = f'/subscriptions/{subscription_id}/resourceGroups/{resource_group_name}/providers/Microsoft.CognitiveServices/accounts/{ai_services_name}/privateEndpointConnections/{pec_name}'
    status_out = utils.run(
        f'az network private-endpoint-connection show --id {pec_id} --query "properties.privateLinkServiceConnectionState.status" -o tsv',
        f'Retrieved PE connection status', 'Failed to retrieve PE connection status')
    current_status = status_out.text.strip() if status_out.success and status_out.text else ''
    if current_status == 'Approved':
        utils.print_ok(f'PE connection {pec_name} is already Approved (auto-approved by Bicep deployment script).')
    else:
        utils.print_info(f'PE connection {pec_name} is {current_status}. Approving now (fallback)...')
        utils.run(
            f'az network private-endpoint-connection approve --id {pec_id} --description "Approved by Foundry E2E Private lab notebook"',
            'SPL PE connection approved', 'Failed to approve SPL PE connection')
else:
    utils.print_error('No PE connection from the Search SPL found on the Foundry account after 10 minutes.')

<a id='4'></a>
### 4️⃣ 🧪 Test the APIM AI Gateway with an APIM subscription key

Because the Foundry account, APIM, and the cross-region OpenAI account are all behind private endpoints, the test calls below need to run from a host that is inside (or peered with) the lab VNet — typically the jump-box VM created by this lab.

> **Easiest path**: the bootstrap drops `Test-AI-Gateway-Primary.ps1` (and `Test-AI-Gateway-CrossRegion.ps1` when the cross-region OpenAI is enabled) onto the jump-box's Public Desktop. Each script has the APIM gateway URL, API path, model deployment name, API version, **and a scoped APIM subscription key** baked in by the deployment, so they don't need `az login` or any Azure AD identity to run. After RDPing in via Bastion, right-click either script → **Run with PowerShell**. The scripts are ACL-locked to Administrators + SYSTEM because they contain the APIM key in cleartext.

**How auth works**
- Caller (jump-box) → APIM with header `Ocp-Apim-Subscription-Key: <scoped APIM key>`
- APIM verifies the subscription, then uses **its own system-assigned managed identity** to obtain a token for the Azure OpenAI backend (see [policy.xml](policy.xml))
- APIM forwards the request to Foundry / cross-region OpenAI over the private endpoint

The APIM API was deployed with `subscriptionRequired: true`, and a dedicated subscription scoped to that single API was provisioned for the test scripts. The Foundry project's `apim-gateway` connection continues to use APIM's master subscription key, whose scope `/` covers all APIs.

If you'd rather drive APIM yourself from the jump-box, the equivalent Python is:

```python
# pip install openai
import os
from openai import AzureOpenAI

client = AzureOpenAI(
    api_key='<APIM subscription key>',          # scoped to /apis/azure-openai
    api_version='2024-10-21',
    base_url='https://<apim-name>.azure-api.net/openai',
)

completion = client.chat.completions.create(
    model='gpt-4o-mini',                         # the Foundry model deployment name
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant routed through the APIM AI Gateway.'},
        {'role': 'user',   'content': 'Tell me one fun fact about Azure API Management.'},
    ],
)
print(completion.choices[0].message.content)
```

All traffic flows: **jump-box → APIM private endpoint → APIM → Foundry private endpoint**.

<a id='5'></a>
### 5️⃣ 🧪 Test the cross-region OpenAI through APIM

Same pattern as above, but the cross-region API in APIM is mounted at `openai-<region>` (e.g. `openai-eastus2`) and exposes the cross-region model deployment.

> The desktop script `Test-AI-Gateway-CrossRegion.ps1` on the jump-box already exercises this path with the values baked in at deploy time, including a dedicated APIM subscription key scoped to the cross-region API.

```python
client = AzureOpenAI(
    api_key='<APIM cross-region subscription key>',
    api_version='2024-10-21',
    base_url='https://<apim-name>.azure-api.net/openai-eastus2',
)
completion = client.chat.completions.create(model='gpt-4o', messages=[...])
```

The request flows: **jump-box → APIM private endpoint → APIM → cross-region OpenAI private endpoint** (in the secondary region, reachable from the primary VNet through its own private endpoint). APIM authenticates to the cross-region OpenAI account with its managed identity (granted the `Cognitive Services OpenAI User` role on that account by the Bicep deployment).

<a id='6'></a>
### 6️⃣ 🖥️ Connect to the jump-box via Azure Bastion

Run from your local machine to open an RDP-over-Bastion tunnel:

```bash
# get the resource IDs
RG=<resource_group_name>
VM_ID=$(az vm show -g $RG -n <jumpbox_name> --query id -o tsv)

# Native client RDP tunnel (Windows jump-box)
az network bastion rdp \
  --name <bastion_name> \
  --resource-group $RG \
  --target-resource-id $VM_ID
```

Or in the Azure Portal: navigate to the jump-box VM → **Connect** → **Bastion**, then sign in with `azureuser` and the password you set in step 0.

When `install_dev_tools = True` (the default) the jump-box is pre-loaded with Python 3.12, Azure CLI, Git, VS Code, PowerShell 7 and Windows Terminal, and the `Azure-Samples/AI-Gateway` repo is cloned to `C:\Git\AI-Gateway` with `requirements.txt` already installed. A desktop shortcut **AI-Gateway Lab** opens the lab folder in VS Code. Bootstrap runs on first boot and takes ~5-10 minutes after the VM shows `Succeeded`; if anything is missing, check `C:\bootstrap.log` and re-run the script if needed.

From the jump-box, sign in (`az login`), open the lab notebook in VS Code, and run the snippet from step 4.

<a id='clean'></a>
### 🗑️ Clean up resources

When you are finished with the lab, run the [clean-up notebook](clean-up-resources.ipynb) to delete the resource group and avoid extra charges.